# Análisis de Calidad: Cartera y Pagos - Paigo
**Autor:** Franco Robotti  
**Objetivo:** Carga, normalización y validación de datasets para el análisis de riesgo crediticio.



### 1. Importación y Carga de Datos
*   **Librerías:** Se utiliza `duckdb` para consultas SQL de alto rendimiento sobre los DataFrames de `pandas`.
*   **Normalización:** Se fuerza el formato `datetime` en las columnas de fechas para asegurar la consistencia en los cálculos posteriores de mora.
*   **Fecha de corte:** Se define una única constante `FECHA_CORTE` como fecha de análisis, reutilizada en todos los chequeos para evitar desincronización.

In [17]:
import os
import duckdb
import numpy as np
import openpyxl
import pandas as pd

# Fecha de análisis (corte)
FECHA_CORTE_STR = "2026-05-14"
FECHA_CORTE = pd.to_datetime(FECHA_CORTE_STR)

In [18]:
file_path = '../data/raw/datasets_riesgo_v2.xlsx'

df_cartera = pd.read_excel(file_path, sheet_name='cartera')
df_pagos = pd.read_excel(file_path, sheet_name='pagos')

df_pagos['fecha_vencimiento'] = pd.to_datetime(df_pagos['fecha_vencimiento'])
df_pagos['fecha_pago_real'] = pd.to_datetime(df_pagos['fecha_pago_real'])
df_cartera['fecha_originacion'] = pd.to_datetime(df_cartera['fecha_originacion'])

print(f"Cartera cargada: {df_cartera.shape[0]} registros.")
print(f"Pagos cargados: {df_pagos.shape[0]} registros.")

Cartera cargada: 400 registros.
Pagos cargados: 1349 registros.


## 2. Auditoría y Calidad de Datos

Antes de proceder con el análisis financiero y de riesgo, es imperativo validar la robustez estructural del dataset. Esta fase garantiza que las conclusiones obtenidas se basen en datos fiables, íntegros y consistentes.

**Alcance de la Auditoría:**

*   **Integridad Referencial:** Validación de las relaciones entre tablas para asegurar la ausencia de registros huérfanos.
*   **Consistencia Temporal:** Verificación de la cronología lógica entre fechas de originación, vencimiento y pago.
*   **Auditoría de Duplicados:** Confirmación de la unicidad de cada préstamo y pago en el sistema.
*   **Análisis de Integridad (Nulos):** Evaluación de datos faltantes y su impacto en la lógica de negocio.
*   **Consistencia Interna:** Validación de la coherencia entre el saldo del préstamo, su estado actual y el comportamiento de pago real.

### 2.1 Integridad Referencial
Se ha validado la relación entre `df_pagos` y `df_cartera`.

> **Resultado:** Se confirma una **integridad del 100%**. Todos los registros de pago están correctamente vinculados a un préstamo existente, garantizando la ausencia de datos huérfanos.

In [19]:
pagos_sin_prestamo = """
SELECT 
    p.id_prestamo,
    p.monto_pagado,
    p.fecha_vencimiento
FROM df_pagos p
LEFT JOIN df_cartera c ON p.id_prestamo = c.id_prestamo
WHERE c.id_prestamo IS NULL
"""

df_pagos_sin_prestamo = duckdb.query(pagos_sin_prestamo).df()

total_pagos = len(df_pagos)
pagos_huerfanos = len(df_pagos_sin_prestamo)
integridad = ((total_pagos - pagos_huerfanos) / total_pagos) * 100

print(f"Integridad Referencial: {integridad:.1f}%")
df_pagos_sin_prestamo

Integridad Referencial: 100.0%


,id_prestamo,monto_pagado,fecha_vencimiento


### 2.2 Consistencia Cronológica
Verificamos varias reglas temporales que deberían cumplirse siempre: ningún pago ni vencimiento puede ser anterior a la originación del crédito, y ninguna fecha de pago puede ubicarse en el futuro respecto al corte.

> **Resultado:** Integridad cronológica confirmada. Los tres controles arrojan **0 casos**, lo que valida la calidad temporal de la información para el análisis de riesgo.

In [20]:
chequeos_cronologicos = duckdb.query(f"""
SELECT 
    'Pago anterior a la originación' AS chequeo, COUNT(*) AS casos
FROM df_pagos p 
    JOIN df_cartera c ON p.id_prestamo = c.id_prestamo
WHERE p.fecha_pago_real < c.fecha_originacion

UNION ALL

SELECT 
    'Vencimiento anterior a la originación' AS chequeo, COUNT(*) AS casos
FROM df_pagos p 
    JOIN df_cartera c ON p.id_prestamo = c.id_prestamo
WHERE p.fecha_vencimiento < c.fecha_originacion

UNION ALL

SELECT 
    'Fecha de pago en el futuro (> corte)' AS chequeo, COUNT(*) AS casos
FROM df_pagos
WHERE fecha_pago_real > '{FECHA_CORTE_STR}'
""").df()

chequeos_cronologicos

,chequeo,casos
0,Pago anterior a la originación,0
1,Vencimiento anterior a la originación,0
2,Fecha de pago en el futuro (> corte),0


### 2.3 Auditoría de Duplicados
Verificamos que no existan registros duplicados que puedan inflar las métricas de cartera o distorsionar los cálculos de riesgo.

**Proceso de verificación:**
* **Detección:** Buscamos registros donde coincida el identificador único (`id_prestamo` en cartera y `id_pago` en pagos).
* **Validación:** Confirmamos la unicidad de cada registro para garantizar la precisión de los cálculos financieros.

> **Resultado:** No se encontraron IDs duplicados ni en cartera ni en pagos, lo que indica que cada préstamo y cada pago están representados de manera única. Esto es crucial para la precisión del análisis posterior, ya que los duplicados distorsionarían los resultados.

In [21]:
# Duplicados en cartera (id_prestamo) y en pagos (id_pago), en variables separadas
df_dup_cartera = duckdb.query("""
SELECT id_prestamo, COUNT(*) AS conteo
FROM df_cartera GROUP BY id_prestamo HAVING COUNT(*) > 1
""").df()

df_dup_pagos = duckdb.query("""
SELECT id_pago, COUNT(*) AS conteo
FROM df_pagos GROUP BY id_pago HAVING COUNT(*) > 1
""").df()

print(f"IDs de préstamo duplicados en cartera: {len(df_dup_cartera)}")
print(f"IDs de pago duplicados en pagos      : {len(df_dup_pagos)}")
display(df_dup_cartera, df_dup_pagos)

IDs de préstamo duplicados en cartera: 0
IDs de pago duplicados en pagos      : 0


,id_prestamo,conteo


,id_pago,conteo


### 2.4. Análisis de Integridad y Valores Nulos
Tras verificar la unicidad de los registros, procedemos a analizar la presencia de valores nulos en `fecha_pago_real`. Estos valores no representan errores de carga, sino información latente sobre el comportamiento de pago.

#### 2.4.1. Diagnóstico de Inconsistencias
Al cruzar los datos de la cartera con el histórico de pagos, detectamos préstamos que, aun presentando cuotas impagas vencidas, mantienen un estado de "Al día" en el sistema original.

> **Hallazgo Clave:** Identificamos 60 préstamos con inconsistencias críticas. Estos casos presentan cuotas impagas desde años anteriores (2022-2024) que no están siendo reflejadas en el estado de mora del sistema.

In [22]:
print("Nulos en Cartera:")
print(df_cartera.isnull().sum())

print("\nNulos en Pagos:")
print(df_pagos.isnull().sum())

Nulos en Cartera:
id_prestamo          0
id_cliente           0
producto             0
fecha_originacion    0
monto_original       0
plazo_meses          0
tna                  0
segmento_cliente     0
canal_adquisicion    0
pais                 0
estado_actual        0
dias_mora            0
saldo_capital        0
dtype: int64

Nulos en Pagos:
id_pago                0
id_prestamo            0
nro_cuota              0
fecha_vencimiento      0
monto_cuota            0
fecha_pago_real      151
monto_pagado           0
dias_atraso          151
dtype: int64


In [23]:
# Inspección visual: muestra de cuotas impagas (fecha_pago_real nula)
nulos = """
SELECT 
    p.id_prestamo, p.nro_cuota, p.fecha_vencimiento, p.monto_cuota,
    c.estado_actual, c.dias_mora
FROM df_pagos AS p
JOIN df_cartera c ON p.id_prestamo = c.id_prestamo
WHERE p.fecha_pago_real IS NULL
ORDER BY p.fecha_vencimiento ASC
LIMIT 10
"""
duckdb.query(nulos).df()

,id_prestamo,nro_cuota,fecha_vencimiento,monto_cuota,estado_actual,dias_mora
0,10366,1,2022-02-03,1383.75,Al día,0
1,10308,1,2022-02-16,3605.33,Cancelado,0
2,10366,2,2022-03-05,1383.75,Al día,0
3,10205,1,2022-03-11,7995.00,31-60 días mora,56
4,10194,1,2022-03-14,1787.83,Al día,0
5,10143,1,2022-03-15,210.33,Al día,0
6,10122,2,2022-03-23,4120.00,31-60 días mora,38
7,10261,1,2022-03-29,3075.00,31-60 días mora,33
8,10320,2,2022-04-11,891.43,Cancelado,0
9,10336,1,2022-04-17,5098.50,Al día,0


In [24]:
inconsistencia_mora = f"""
SELECT 
    c.id_prestamo,
    c.estado_actual,
    c.dias_mora,
    COUNT(p.nro_cuota) AS cuotas_vencidas_no_pagas,
    MIN(p.fecha_vencimiento) AS vencimiento_mas_antiguo
FROM df_cartera c
JOIN df_pagos p ON c.id_prestamo = p.id_prestamo
WHERE p.fecha_pago_real IS NULL 
  AND p.fecha_vencimiento < '{FECHA_CORTE_STR}'
  AND c.estado_actual = 'Al día'
GROUP BY ALL
"""

df_alarmas = duckdb.query(inconsistencia_mora).df()
print(f"Total de préstamos con inconsistencia: {len(df_alarmas)}")
df_alarmas

Total de préstamos con inconsistencia: 60


,id_prestamo,estado_actual,dias_mora,cuotas_vencidas_no_pagas,vencimiento_mas_antiguo
0,10076,Al día,0,3,2022-11-19
1,10202,Al día,0,1,2022-10-15
2,10230,Al día,0,2,2024-01-21
3,10176,Al día,0,1,2023-10-27
4,10318,Al día,0,1,2023-11-15
5,10326,Al día,0,1,2024-02-02
6,10363,Al día,0,2,2022-07-14
7,10046,Al día,0,1,2023-08-16
8,10107,Al día,0,1,2024-03-22
9,10143,Al día,0,4,2022-03-15


#### 2.4.2. Clasificación y Cuantificación de Nulos
**Análisis de Hallazgos:**
* **Mora Real Reconocida (73 registros):** Representan el 48.3% de los nulos. Son registros con consistencia entre tablas, donde la cuota está vencida y el préstamo correctamente identificado en la cartera. El capital asociado es de $343,422.93.
* **Inconsistencias de Mora (78 registros):** Representan el 51.7% de los nulos. Son cuotas vencidas (desde febrero de 2022) no pagadas, pero cuyo préstamo figura erróneamente como "Al día". Esta anomalía afecta a 60 préstamos únicos y compromete un capital de $263,668.14 que no está siendo provisionado correctamente.
* **Ausencia de Cuotas Futuras:** Se confirmó que 0 registros corresponden a cuotas vigentes o futuras, lo que indica que el dataset es una fotografía de obligaciones exigibles que permanecen impagas.

In [25]:
analisis_restante_nulos = f"""
SELECT 
    CASE 
        WHEN p.fecha_vencimiento >= '{FECHA_CORTE_STR}' THEN 'Cuotas Futuras (Vigentes)'
        WHEN p.fecha_vencimiento < '{FECHA_CORTE_STR}' AND c.estado_actual != 'Al día' THEN 'Mora Correcta'
        WHEN p.fecha_vencimiento < '{FECHA_CORTE_STR}' AND c.estado_actual = 'Al día' THEN 'Inconsistencia'
        ELSE 'Otros'
    END AS tipo_nulo,
    COUNT(*) AS cantidad,
    SUM(p.monto_cuota) AS capital_asociado,
    MIN(p.fecha_vencimiento) AS vencimiento_min,
    MAX(p.fecha_vencimiento) AS vencimiento_max,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) || '%' AS porcentaje
FROM df_pagos p
LEFT JOIN df_cartera c ON p.id_prestamo = c.id_prestamo
WHERE p.fecha_pago_real IS NULL
GROUP BY 1
ORDER BY 1
"""

df_detalle_151 = duckdb.query(analisis_restante_nulos).df()
df_detalle_151

,tipo_nulo,cantidad,capital_asociado,vencimiento_min,vencimiento_max,porcentaje
0,Inconsistencia,78,263668.14,2022-02-03,2024-10-31,51.66%
1,Mora Correcta,73,343422.93,2022-02-16,2024-12-03,48.34%


#### 2.4.3. Validación de los montos pagados
Más allá de los nulos, controlamos la coherencia del importe pagado contra la cuota pactada: sobrepagos, pagos con monto cero (que deberían ser nulos) y pagos parciales.

In [26]:
validacion_montos = duckdb.query("""
SELECT 
    SUM(CASE WHEN fecha_pago_real IS NOT NULL AND monto_pagado > monto_cuota THEN 1 ELSE 0 END) AS sobrepagos,
    SUM(CASE WHEN fecha_pago_real IS NOT NULL AND monto_pagado = 0 THEN 1 ELSE 0 END) AS pagados_en_cero,
    SUM(CASE WHEN fecha_pago_real IS NOT NULL AND monto_pagado < monto_cuota THEN 1 ELSE 0 END) AS pagos_parciales
FROM df_pagos
""").df()
display(validacion_montos)

cuotas_pagadas = int(df_pagos['fecha_pago_real'].notna().sum())
parciales = int(validacion_montos['pagos_parciales'][0])
print(f"Cuotas pagadas: {cuotas_pagadas} | parciales: {parciales} ({parciales/cuotas_pagadas*100:.1f}%)")

,sobrepagos,pagados_en_cero,pagos_parciales
0,0.0,0.0,1198.0


Cuotas pagadas: 1198 | parciales: 1198 (100.0%)


> **Hallazgo a comunicar:** no hay sobrepagos ni cuotas con monto cero pero con fecha (no hay contradicciones duras). Sin embargo, **la totalidad de las cuotas pagadas (1.198 de 1.198, el 100%)** registran `monto_pagado < monto_cuota`. Que *ningún* pago alcance la cuota pactada es demasiado sistemático para ser azar: lo más probable es que `monto_pagado` mida solo una porción del importe (p. ej. capital sin interés) y no un pago parcial real. Esto **no es un error de carga**, pero abre una pregunta de negocio a confirmar con el equipo de datos. Hasta entonces, una cuota con fecha de pago se considera **pagada** a efectos de la mora.

### 2.5. Validación de Consistencia Interna
Verificamos la coherencia entre las variables de saldo y el estado del préstamo en la cartera.

#### 2.5.1. Hallazgos de Consistencia
Realizamos dos pruebas de integridad sobre el dataset:
* **Consistencia de Mora:** Verificamos si existen préstamos con `dias_mora > 0` marcados como "Al día".
* **Consistencia de Saldo:** Buscamos préstamos con `saldo_capital = 0` que no se encuentren en estado "Cancelado".

> **Resultado:** Ambas pruebas arrojan 0 casos de error, confirmando que la tabla `cartera` es consistente **consigo misma**. Esto refuerza que el problema detectado antes no es un error de formato, sino una **discrepancia entre procesos de negocio**: la cartera es internamente coherente, pero no refleja la realidad de los pagos, porque ignoró las cuotas vencidas e impagas de esos 60 préstamos.

In [27]:
inconsistencia_campos_cartera = """
SELECT 
    id_prestamo, pais, producto, estado_actual, dias_mora, saldo_capital
FROM df_cartera
WHERE dias_mora > 0 
  AND estado_actual = 'Al día'
"""

df_error_campos = duckdb.query(inconsistencia_campos_cartera).df()
print(f"Casos con dias_mora > 0 y estado 'Al día': {len(df_error_campos)}")
df_error_campos

Casos con dias_mora > 0 y estado 'Al día': 0


,id_prestamo,pais,producto,estado_actual,dias_mora,saldo_capital


In [28]:
inconsistencia_saldo_final = """
SELECT c.*
FROM df_cartera AS c
WHERE ROUND(saldo_capital, 2) = 0 
  AND estado_actual NOT IN ('Cancelado')
"""

df_errores_saldo = duckdb.query(inconsistencia_saldo_final).df()
print(f"Hallazgo: {len(df_errores_saldo)} préstamos que no deben nada pero siguen activos.")
df_errores_saldo

Hallazgo: 0 préstamos que no deben nada pero siguen activos.


,id_prestamo,id_cliente,producto,fecha_originacion,monto_original,plazo_meses,tna,segmento_cliente,canal_adquisicion,pais,estado_actual,dias_mora,saldo_capital


#### 2.5.2. Definición de la "Fuente de Verdad"
Para proceder con la corrección de los datos, establecemos los siguientes criterios técnicos:
* **Fuente de Verdad:** La tabla de pagos es la fuente de verdad absoluta para determinar la mora real.
* **Criterio de Mora Efectiva:** Se define como mora cualquier préstamo con al menos una cuota de vencimiento anterior al corte (`FECHA_CORTE = 2026-05-14`) y `fecha_pago_real` nula.
* **Hipótesis de Error:** Los 60 casos inconsistentes se asumen como una falla en la actualización automática de los procesos del sistema.

> **Nota sobre la fecha de corte:** el último vencimiento del dataset es 2025-03-03, mientras que el corte de análisis es 2026-05-14 (~14 meses después). Por eso no hay cuotas vigentes/futuras y los días de atraso de las impagas son elevados (se miden hasta el corte). Es coherente con una fotografía tomada en mayo de 2026; si se quisiera evaluar la mora "a la fecha del último dato", bastaría con mover `FECHA_CORTE`.

In [29]:
# PASO 1: identificar impagas con un flag EXPLÍCITO, sobre una COPIA (no mutamos df_pagos)
df_pagos_calc = df_pagos.copy()
df_pagos_calc['impago'] = df_pagos_calc['fecha_pago_real'].isna()

# Días de atraso reales por cuota: para las impagas, días vencidos hasta el corte
dias_vencidos = (FECHA_CORTE - df_pagos_calc['fecha_vencimiento']).dt.days.clip(lower=0)
df_pagos_calc['dias_atraso'] = df_pagos_calc['dias_atraso'].fillna(dias_vencidos)

# PASO 2: agregación de la mora real sobre las impagas vencidas (filtro por el flag, no por fecha sentinela)
mora_por_prestamo = duckdb.query(f"""
    SELECT 
        id_prestamo,
        COUNT(*) AS cuotas_vencidas,
        MAX(dias_atraso) AS dias_mora_calculados,
        SUM(monto_cuota) AS capital_en_mora
    FROM df_pagos_calc
    WHERE impago = TRUE
      AND fecha_vencimiento < '{FECHA_CORTE_STR}'
    GROUP BY id_prestamo
""").df()

# PASO 3: cruce y reclasificación
df_cartera_final = df_cartera.merge(mora_por_prestamo, on='id_prestamo', how='left')
df_cartera_final['cuotas_vencidas'] = df_cartera_final['cuotas_vencidas'].fillna(0)
df_cartera_final['dias_mora_calculados'] = df_cartera_final['dias_mora_calculados'].fillna(0)
df_cartera_final['capital_en_mora'] = df_cartera_final['capital_en_mora'].fillna(0)

df_cartera_final['estado_analista'] = np.where(
    df_cartera_final['cuotas_vencidas'] > 0,
    'Mora Reclasificada',
    df_cartera_final['estado_actual']
)

reclasificados = int((df_cartera_final['estado_analista'] == 'Mora Reclasificada').sum())
print(f"Préstamos con mora reclasificada: {reclasificados}")
df_cartera_final.head()

Préstamos con mora reclasificada: 122


,id_prestamo,id_cliente,producto,fecha_originacion,monto_original,plazo_meses,tna,segmento_cliente,canal_adquisicion,pais,estado_actual,dias_mora,saldo_capital,cuotas_vencidas,dias_mora_calculados,capital_en_mora,estado_analista
0,10000,1189,Préstamo Personal,2022-09-08,6900,9,0.48,Recurrente,Alianza Comercial,UY,Al día,0,1326.50,0.0,0.0,0.00,Al día
1,10001,1050,Adelanto de Sueldo,2022-08-12,1000,1,0.30,Recurrente,Alianza Comercial,AR,Al día,0,123.88,0.0,0.0,0.00,Al día
2,10002,1087,BNPL,2024-04-08,14600,3,0.36,Nuevo,Paid Social,UY,31-60 días mora,36,10633.55,0.0,0.0,0.00,31-60 días mora
3,10003,1088,BNPL,2022-04-15,28800,3,0.36,Nuevo,Alianza Comercial,AR,Al día,0,12184.75,0.0,0.0,0.00,Al día
4,10004,1160,Préstamo Personal,2022-05-08,59700,18,0.48,Inactivo,Alianza Comercial,AR,Al día,0,21722.47,2.0,1407.0,6898.66,Mora Reclasificada


In [30]:
# PASO 4: EXPORTACIÓN A EXCEL
os.makedirs("../data/clean", exist_ok=True)
archivo_salida = "../data/clean/cartera_consolidada_limpia.xlsx"
df_cartera_final.to_excel(archivo_salida, index=False)
print(f"¡Éxito! El tablón analítico limpio se exportó correctamente como: {archivo_salida}")

¡Éxito! El tablón analítico limpio se exportó correctamente como: ../data/clean/cartera_consolidada_limpia.xlsx
